In [10]:
import os
import mlflow

TRACKING_URI = os.getenv(
    "MLFLOW_TRACKING_URI",
    "MLFLOW_TRACKING_URI_LOCAL",
    # mlflow.set_tracking_uri("http://127.0.0.1:5000")
)

mlflow.set_tracking_uri(TRACKING_URI)

with mlflow.start_run():
    mlflow.log_param("param1", 15)
    mlflow.log_metric("metric1", 0.89)

🏃 View run mercurial-rat-523 at: https://mlflow-dashboard.duckdns.org/#/experiments/5/runs/919d558b465d49fca0532f30aebce48a
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/5


## Perform the Same EDA

In [90]:
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [91]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

class TextCleaner:
    """Class to clean and preprocess raw text comments."""
    def __init__(self):
        self._ensure_nltk_resources()
        self.stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
        self.lemmatizer = WordNetLemmatizer()
        
    def _ensure_nltk_resources(self):
        try:
            nltk.data.find('corpora/stopwords')
        except LookupError:
            nltk.download('stopwords', quiet=True)
        try:
            nltk.data.find('corpora/wordnet')
        except LookupError:
            nltk.download('wordnet', quiet=True)
            
    def clean_text(self, text):
        if not isinstance(text, str):
            return ""
            
        # Convert to lowercase
        text = text.lower()
        
        # Remove trailing and leading whitespaces
        text = text.strip()
        
        # Remove newline characters
        text = re.sub(r'\n', ' ', text)
        
        # Remove URLs
        url_pattern = r"http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+"
        text = re.sub(url_pattern, '', text)
        
        # Remove non-alphanumeric characters, except punctuation
        text = re.sub(r'[^A-Za-z0-9\s!?.,]', '', text)
        
        # Remove stopwords
        text = ' '.join(word for word in text.split() if word not in self.stop_words)
        
        # Lemmatize words
        text = ' '.join(self.lemmatizer.lemmatize(word) for word in text.split())
        
        return text

class DataPreprocessor:
    """Class to handle DataFrame level data preprocessing."""
    def __init__(self, text_column='clean_comment'):
        self.text_column = text_column
        self.text_cleaner = TextCleaner()
        
    def fit(self, df, y=None):
        return self
        
    def transform(self, df):
        """Executes the full preprocessing pipeline on the input DataFrame."""
        df_processed = df.copy()
        
        if self.text_column not in df_processed.columns:
            return df_processed
            
        # 1. Drop missing values based on text column
        df_processed.dropna(subset=[self.text_column], inplace=True)
        
        # 2. Drop duplicates
        df_processed.drop_duplicates(inplace=True)
        
        # 3. Strip initial empty records before further processing
        df_processed = df_processed[df_processed[self.text_column].str.strip() != ""]
        
        # 4. Apply text cleaning
        df_processed[self.text_column] = df_processed[self.text_column].apply(self.text_cleaner.clean_text)
        
        # 5. Remove any empty strings formed after cleaning
        df_processed = df_processed[df_processed[self.text_column].str.strip() != ""]
        
        return df_processed


In [92]:
import sys
import os

# Add the parent directory so Python can find the 'src' module
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.data_preprocessing import DataPreprocessor

# 1. Load your raw data
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')

# 2. Initialize the preprocessor and transform the data
preprocessor = DataPreprocessor(text_column='clean_comment')
df_clean = preprocessor.transform(df)

# Check the fully cleaned results
df_clean.head()


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [14]:
df_clean.shape

(36662, 2)

In [15]:
df_clean.isnull().sum()

clean_comment    0
category         0
dtype: int64

In [93]:
# Ensure the directory exists first
os.makedirs('data/processed', exist_ok=True)
output_path = 'data/processed/cleaned_data.csv'
df_clean.to_csv(output_path, index=False)
print(f"Cleaned data saved successfully to: {output_path}")
# Check the results
df_clean.head()

Cleaned data saved successfully to: data/processed/cleaned_data.csv


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [ ]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Vectorization

### TF-IDF (Term Frequency - Inverse Document Frequency)

> TF-IDF is an evolution of the Bag of Words model designed to solve the problem of highly frequent, low-meaning words.

#### How it works: It calculates a weight for each word using two metrics:

1. TF (Term Frequency): How often a word appears in a single document.

2. IDF (Inverse Document Frequency): How rare the word is across the entire corpus of documents.

3. The formula multiplies them (TF * IDF). If a word is frequent in one document but rare globally, it gets a high score (meaning it's an important keyword for that document). If it's frequent everywhere (like "is" or "and"), it gets a score close to zero.

- Vector type: Sparse matrix (but with continuous float values instead of integer counts).

- Pros: Excellent for keyword extraction and document search. It heavily penalizes useless stop words naturally.

- Cons: Still creates large sparse matrices. Like BoW, it cannot capture context, word order, or semantic meaning.

### Bag of Words (BoW) & CountVectorizer

> It is important to understand that Bag of Words is the theoretical concept, while CountVectorizer is the practical implementation of this concept (specifically in Python's scikit-learn library).

- How it works: It creates a vocabulary of all unique words in the entire corpus (dataset). For each document, it counts how many times each word from the vocabulary appears. It completely ignores the order of words and grammar.

    - Vector type: Sparse matrix (mostly zeros).
    - Pros: Very simple to understand and fast to compute.
    - Cons: * Creates massive, memory-heavy sparse matrices.

- Treats all words equally (a common word like "the" will dominate the counts, even though it carries little meaning).
- Zero semantic understanding (e.g., "happy" and "joyful" are treated as completely unrelated dimensions).

In [11]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df_clean["clean_comment"]).toarray()
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(36662, 10000))

In [12]:
X.shape

(36662, 10000)

In [16]:
y = df_clean["category"]

In [14]:
class ModelTrainer:
    def __init__(
        self,
        experiment_name="Sentiment_Classification",
        text_column="clean_comment",
        target_column="category",
        test_size=0.2,
        random_state=42,
        max_features=8000
    ):
        self.experiment_name = experiment_name
        self.text_column = text_column
        self.target_column = target_column
        self.test_size = test_size
        self.random_state = random_state
        self.max_features = max_features

        self.vectorizer = TfidfVectorizer(max_features=self.max_features)
        self.model = RandomForestClassifier(
            n_estimators=200,
            random_state=self.random_state
        )
        # mlflow.set_tracking_uri("https://mlflow-dashboard.duckdns.org")
        mlflow.set_tracking_uri("http://127.0.0.1:5000")
        mlflow.set_experiment(self.experiment_name)

    def train(self, df: pd.DataFrame):

        X = df[self.text_column]
        y = df[self.target_column]

        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )

        # Vectorization
        X_train_vec = self.vectorizer.fit_transform(X_train)
        X_test_vec = self.vectorizer.transform(X_test)

        with mlflow.start_run():

            # Log parameters
            mlflow.log_param("model", "RandomForest")
            mlflow.log_param("vectorizer", "TF-IDF")
            mlflow.log_param("max_features", self.max_features)
            mlflow.log_param("test_size", self.test_size)

            # Train
            self.model.fit(X_train_vec, y_train)

            # Predict
            y_pred = self.model.predict(X_test_vec)

            # Metrics
            accuracy = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)

            mlflow.log_metric("accuracy", accuracy)

            for label, metrics in report.items():
                if isinstance(metrics, dict):
                    for metric_name, value in metrics.items():
                        mlflow.log_metric(f"{label}_{metric_name}", value)

            # Confusion Matrix
            cm = confusion_matrix(y_test, y_pred)

            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
            plt.title("Confusion Matrix")
            plt.ylabel("Actual")
            plt.xlabel("Predicted")

            os.makedirs("artifacts", exist_ok=True)
            cm_path = "artifacts/confusion_matrix.png"
            plt.savefig(cm_path)
            plt.close()

            mlflow.log_artifact(cm_path)

            # Log model
            mlflow.sklearn.log_model(self.model, "random_forest_model")

            print(f"Accuracy: {accuracy:.4f}")
            print(classification_report(y_test, y_pred))

        return self.model, self.vectorizer

In [ ]:
trainer = ModelTrainer()
model, vectorizer = trainer.train(df_clean)

## Import Cleaned Data:

In [110]:
import pandas as pd

# 1. Load the data
df_clean = pd.read_csv('data/processed/cleaned_data.csv')

# 2. Map categories (-1 -> 2, 1 -> 1, 0 -> 0)
df_clean['category'] = df_clean['category'].map({-1: 2, 1: 1, 0: 0})

# 3. Check results
df_clean.head()


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,2
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [111]:
y = df_clean["category"]
y

0        1
1        1
2        2
3        0
4        1
        ..
36657    0
36658    1
36659    0
36660    1
36661    0
Name: category, Length: 36662, dtype: int64

In [112]:
df_clean.isnull().sum()

clean_comment    0
category         0
dtype: int64

### 🧠 Architecture Idea (what you actually want)

We split into 3 layers:

#### 1. Vectorizer Strategy (pluggable)

* TF-IDF
* CountVectorizer

#### 2. Model Strategy (pluggable)

* RandomForest
* XGBoost
* LightGBM

#### 3. Trainer (orchestrator)

* handles MLflow
* training loop
* evaluation

---

#### 🏗️ Design Pattern: Strategy Pattern

We’ll implement:

```
VectorizerFactory
ModelFactory
Trainer (uses both)
```

#### 🧠 Why this design is correct

#### ✔ SOLID principles applied

#### 1. Open/Closed Principle

You can add new models without touching trainer

#### 2. Strategy Pattern

Model + vectorizer are interchangeable

#### 3. Separation of concerns

* Factory → creation
* Trainer → logic
* MLflow → tracking


In [ ]:
# import numpy as np
# from gensim.models import Word2Vec
# from sklearn.base import BaseEstimator, TransformerMixin


# class Word2VecVectorizer(BaseEstimator, TransformerMixin):

#     def __init__(self, vector_size=100, window=5, min_count=2):
#         self.vector_size = vector_size
#         self.window = window
#         self.min_count = min_count
#         self.model = None

#     def fit(self, X, y=None):

#         tokenized = [text.split() for text in X]

#         self.model = Word2Vec(
#             sentences=tokenized,
#             vector_size=self.vector_size,
#             window=self.window,
#             min_count=self.min_count,
#             workers=4
#         )

#         return self

#     def transform(self, X):

#         return np.array([
#             self._vectorize(text.split())
#             for text in X
#         ])

#     def _vectorize(self, words):

#         vectors = [
#             self.model.wv[w]
#             for w in words
#             if w in self.model.wv
#         ]

#         if len(vectors) == 0:
#             return np.zeros(self.vector_size)

#         return np.mean(vectors, axis=0)

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


class VectorizerFactory:
    @staticmethod
    def get(name: str, max_features=8000):
        name = name.lower()

        if name == "tfidf":
            return TfidfVectorizer(max_features=max_features)

        elif name == "count":
            return CountVectorizer(max_features=max_features)
        
        # elif name == "word2vec":
        #     return Word2VecVectorizer()

        else:
            raise ValueError(f"Unknown vectorizer: {name}")

In [12]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


class ModelFactory:
    @staticmethod
    def get(name: str, random_state=42):

        name = name.lower()

        if name == "random_forest":
            return RandomForestClassifier(
                n_estimators=800,
                max_depth=15,
                random_state=random_state
            )

        elif name == "xgboost":
            return XGBClassifier(
                n_estimators=300,
                learning_rate=0.1,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric="logloss",
                random_state=random_state
            )

        elif name == "lightgbm":
            return LGBMClassifier(
                n_estimators=300,
                learning_rate=0.1,
                random_state=random_state
            )

        else:
            raise ValueError(f"Unknown model: {name}")

In [115]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


class ModelTrainer:

    def __init__(
        self,
        experiment_name="Sentiment_Classification",
        text_column="clean_comment",
        target_column="category",
        test_size=0.2,
        random_state=42
    ):
        self.text_column = text_column
        self.target_column = target_column
        self.test_size = test_size
        self.random_state = random_state

        mlflow.set_tracking_uri("https://mlflow-dashboard.duckdns.org")
        mlflow.set_experiment(experiment_name)

    # ======================
    # SAFE FEATURE CASTING (MODEL-BASED)
    # ======================
    def _cast_features(self, X, model_name):
        if model_name.lower() in ["lightgbm"]:
            return X.astype("float32")
        return X

    # ======================
    # SAFE LABEL CLEANING
    # ======================
    def _clean_labels(self, y):
        return y.astype(str)

    def train(self, df: pd.DataFrame, vectorizer_name="tfidf", model_name="random_forest"):

        # ======================
        # Clean input
        # ======================
        df = df.copy()
        df[self.text_column] = df[self.text_column].fillna("")
        df[self.target_column] = df[self.target_column].fillna("unknown")

        X = df[self.text_column]
        y = self._clean_labels(df[self.target_column])

        # ======================
        # Split
        # ======================
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )

        # ======================
        # Build components
        # ======================
        vectorizer = VectorizerFactory.get(vectorizer_name)
        model = ModelFactory.get(model_name, self.random_state)

        # ======================
        # Vectorization
        # ======================
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        # ======================
        # Model-aware casting
        # ======================
        X_train_vec = self._cast_features(X_train_vec, model_name)
        X_test_vec = self._cast_features(X_test_vec, model_name)

        with mlflow.start_run():

            # ======================
            # Log config
            # ======================
            mlflow.log_param("vectorizer", vectorizer_name)
            mlflow.log_param("model", model_name)

            # ======================
            # Train
            # ======================
            model.fit(X_train_vec, y_train)

            # ======================
            # Predict
            # ======================
            y_pred = model.predict(X_test_vec)

            # ======================
            # Metrics
            # ======================
            acc = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])
            mlflow.log_metric("precision_macro", report["macro avg"]["precision"])
            mlflow.log_metric("recall_macro", report["macro avg"]["recall"])

            # ======================
            # Log model
            # ======================
            mlflow.sklearn.log_model(model, artifact_path="model")

            print(f"Accuracy: {acc:.4f}")
            print(classification_report(y_test, y_pred))

        return model, vectorizer

In [116]:
trainer = ModelTrainer()

In [76]:
trainer.train(df_clean, vectorizer_name="count", model_name="random_forest")

2026/05/09 01:02:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:02:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.6484
              precision    recall  f1-score   support

           0       0.67      0.82      0.74      2529
           1       0.63      0.84      0.72      3154
           2       0.94      0.01      0.02      1650

    accuracy                           0.65      7333
   macro avg       0.75      0.56      0.49      7333
weighted avg       0.71      0.65      0.57      7333

🏃 View run dashing-fly-690 at: https://mlflow-dashboard.duckdns.org/#/experiments/5/runs/4210cd7b9e864c919d2186825233875e
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/5


(RandomForestClassifier(max_depth=15, n_estimators=800, random_state=42),
 CountVectorizer(max_features=8000))

In [117]:
trainer.train(df_clean, vectorizer_name="tfidf", model_name="random_forest")

2026/05/09 01:50:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:50:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.6482
              precision    recall  f1-score   support

           0       0.67      0.81      0.74      2529
           1       0.63      0.85      0.72      3154
           2       0.95      0.01      0.02      1650

    accuracy                           0.65      7333
   macro avg       0.75      0.56      0.49      7333
weighted avg       0.72      0.65      0.57      7333

🏃 View run unequaled-bee-128 at: https://mlflow-dashboard.duckdns.org/#/experiments/5/runs/1298fea1d91f4c638162a33d61efc7f4
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/5


(RandomForestClassifier(max_depth=15, n_estimators=800, random_state=42),
 TfidfVectorizer(max_features=8000))

In [31]:
trainer.train(df_clean, vectorizer_name="tfidf", model_name="xgboost")

2026/05/08 23:51:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 23:51:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.8047
Cleaned data saved at: .\data\processed\cleaned_data.csv
Model saved at: .\artifacts\models\xgboost_tfidf.pkl
              precision    recall  f1-score   support

           0       0.76      0.96      0.84      2529
           1       0.85      0.81      0.83      3154
           2       0.83      0.56      0.67      1650

    accuracy                           0.80      7333
   macro avg       0.81      0.78      0.78      7333
weighted avg       0.81      0.80      0.80      7333

🏃 View run useful-robin-591 at: https://mlflow-dashboard.duckdns.org/#/experiments/5/runs/61ba1fa34fb6480c8c10437e19a46760
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/5


(XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.8, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='logloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.1, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=6, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=300, n_jobs=None,
               num_parallel_tree=None, ...),
 TfidfVectorizer(max_features=8000))

In [50]:
trainer.train(df_clean, vectorizer_name="count", model_name="lightgbm")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.440212 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13568
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 3677
[LightGBM] [Info] Start training from score -1.064557
[LightGBM] [Info] Start training from score -0.843611
[LightGBM] [Info] Start training from score -1.491810


d:\MyFiles\GitHub\YouTube Viewer Sentiment Analysis\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/09 00:39:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 00:39:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.8722
              precision    recall  f1-score   support

           0       0.85      0.97      0.91      2529
           1       0.90      0.87      0.89      3154
           2       0.85      0.72      0.78      1650

    accuracy                           0.87      7333
   macro avg       0.87      0.85      0.86      7333
weighted avg       0.87      0.87      0.87      7333

🏃 View run unique-horse-207 at: https://mlflow-dashboard.duckdns.org/#/experiments/5/runs/9beccae34fb54639b6ecaaeb080eb751
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/5


(LGBMClassifier(n_estimators=300, random_state=42),
 CountVectorizer(max_features=8000))

### Add ngram parameter

In [103]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


class BoWVectorizer:
    def __init__(self, ngram_range=(1, 1), max_features=5000):
        self.model = CountVectorizer(
            ngram_range=ngram_range,
            max_features=max_features
        )

    def fit(self, X):
        return self.model.fit(X)

    def transform(self, X):
        return self.model.transform(X)


class TFIDFVectorizer:
    def __init__(self, ngram_range=(1, 1), max_features=5000):
        self.model = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=max_features
        )

    def fit(self, X):
        return self.model.fit(X)

    def transform(self, X):
        return self.model.transform(X)

In [104]:
class VectorizerFactory:

    @staticmethod
    def get(vectorizer_type, ngram_range, max_features):

        if vectorizer_type == "BoW":
            return BoWVectorizer(ngram_range, max_features)

        elif vectorizer_type == "TF-IDF":
            return TFIDFVectorizer(ngram_range, max_features)

        else:
            raise ValueError(f"Unknown vectorizer: {vectorizer_type}")

In [105]:
from sklearn.ensemble import RandomForestClassifier


class RandomForestModel:
    def __init__(self, n_estimators=200, max_depth=15, random_state=42):
        self.model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state
        )

    def fit(self, X, y):
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)

In [106]:
class ExperimentConfig:
    def __init__(
        self,
        vectorizer_type,
        ngram_range,
        max_features,
        vectorizer_name
    ):
        self.vectorizer_type = vectorizer_type
        self.ngram_range = ngram_range
        self.max_features = max_features
        self.vectorizer_name = vectorizer_name

In [107]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


class ExperimentRunner:

    def __init__(self, df, experiment_name="NLP_Experiments"):
        self.df = df

        mlflow.set_tracking_uri("https://mlflow-dashboard.duckdns.org")
        mlflow.set_experiment(experiment_name)

    def run(self, config: ExperimentConfig):

        X_train, X_test, y_train, y_test = train_test_split(
            self.df["clean_comment"],
            self.df["category"],
            test_size=0.2,
            random_state=42,
            stratify=self.df["category"]
        )

        # ======================
        # Strategy injection
        # ======================
        vectorizer = VectorizerFactory.get(
            config.vectorizer_type,
            config.ngram_range,
            config.max_features
        )

        model = RandomForestModel()

        # ======================
        # Train vectorizer
        # ======================
        X_train_vec = vectorizer.fit(X_train).transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        with mlflow.start_run():

            # ======================
            # Tags
            # ======================
            mlflow.set_tag("vectorizer", config.vectorizer_name)
            mlflow.set_tag("model", "RandomForest")

            # ======================
            # Train model
            # ======================
            model.fit(X_train_vec, y_train)

            y_pred = model.predict(X_test_vec)

            # ======================
            # Metrics
            # ======================
            acc = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])

            # ======================
            # Confusion Matrix
            # ======================
            cm = confusion_matrix(y_test, y_pred)

            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
            plt.title(f"Confusion Matrix: {config.vectorizer_name}")
            plt.savefig("confusion_matrix.png")
            mlflow.log_artifact("confusion_matrix.png")
            plt.close()

            # ======================
            # Log model
            # ======================
            mlflow.sklearn.log_model(model.model, "model")

            print(f"[{config.vectorizer_name}] Accuracy: {acc:.4f}")

In [19]:
# import mlflow

# # 1. Get the experiment details by name
# experiment = mlflow.get_experiment_by_name("NLP_Experiments")

# # 2. Delete it using its experiment_id
# if experiment:
#     mlflow.delete_experiment(experiment.experiment_id)
#     print(f"Experiment '{experiment.name}' deleted successfully.")
# else:
#     print("Experiment not found.")


In [109]:
ngram_ranges = [(1, 1), (1, 2), (1, 3)]
max_features = 5000

runner = ExperimentRunner(df_clean)

for ngram in ngram_ranges:

    runner.run(ExperimentConfig(
        vectorizer_type="BoW",
        ngram_range=ngram,
        max_features=max_features,
        vectorizer_name="BoW"
    ))

    runner.run(ExperimentConfig(
        vectorizer_type="TF-IDF",
        ngram_range=ngram,
        max_features=max_features,
        vectorizer_name="TF-IDF"
    ))

2026/05/09 01:37:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:37:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[BoW] Accuracy: 0.6458
🏃 View run resilient-vole-382 at: https://mlflow-dashboard.duckdns.org/#/experiments/6/runs/e960479ea96948248fb46604e87d3646
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/6


2026/05/09 01:38:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:38:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[TF-IDF] Accuracy: 0.6493
🏃 View run learned-gnat-852 at: https://mlflow-dashboard.duckdns.org/#/experiments/6/runs/1d862790dd334f27bccc1402937dd4fd
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/6


2026/05/09 01:40:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:40:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

[TF-IDF] Accuracy: 0.6546
🏃 View run fortunate-eel-739 at: https://mlflow-dashboard.duckdns.org/#/experiments/6/runs/23331956047547fbbac92dcf7e6f6e4d
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/6


2026/05/09 01:42:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:42:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[BoW] Accuracy: 0.6497
🏃 View run bright-mole-515 at: https://mlflow-dashboard.duckdns.org/#/experiments/6/runs/424f515f1a5042509f4c29668ff67bfc
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/6


2026/05/09 01:43:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 01:43:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[TF-IDF] Accuracy: 0.6533
🏃 View run ambitious-finch-815 at: https://mlflow-dashboard.duckdns.org/#/experiments/6/runs/5a49e24fe67a42e1b4df7305990b25e2
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/6


### Try Max Features 

In [14]:
import pandas as pd

# 1. Load the data
df_clean = pd.read_csv('data/processed/cleaned_data.csv')

# 2. Map categories (-1 -> 2, 1 -> 1, 0 -> 0)
df_clean['category'] = df_clean['category'].map({-1: 2, 1: 1, 0: 0})

# 3. Check results
df_clean.head()


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,2
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


class VectorizerFactory:
    @staticmethod
    def get(name: str, ngram_range=(1, 1), max_features=8000):
        name = name.lower()

        if name == "tfidf":
            return TfidfVectorizer(
                ngram_range=ngram_range,
                max_features=max_features
            )

        elif name == "count":
            return CountVectorizer(
                ngram_range=ngram_range,
                max_features=max_features
            )

        else:
            raise ValueError(f"Unknown vectorizer: {name}")

In [16]:
import os
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


class ModelTrainer:

    def __init__(
        self,
        df,
        text_column="clean_comment",
        target_column="category",
        test_size=0.2,
        random_state=42,
        ngram_range=(1, 3),
        experiment_name="Sentiment_Classification_Max_Features",
    ):
        self.df = df
        self.text_column = text_column
        self.target_column = target_column
        self.test_size = test_size
        self.random_state = random_state
        self.ngram_range = ngram_range
        
        mlflow.set_tracking_uri("https://mlflow-dashboard.duckdns.org")
        mlflow.set_experiment(experiment_name)
    
    def config(self, max_features_list, model_name="random_forest"):

        for max_features in max_features_list:

            self.trainer(
                vectorizer_name="tfidf",
                max_features=max_features,
                model_name=model_name
            )

    def trainer(self, vectorizer_name, max_features, model_name):

        # ======================
        # Split
        # ======================
        X = self.df[self.text_column].fillna("")
        y = self.df[self.target_column]

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )

        # ======================
        # Build components
        # ======================
        vectorizer = VectorizerFactory.get(
            vectorizer_name,
            ngram_range=self.ngram_range,
            max_features=max_features
        )

        model = ModelFactory.get(model_name, self.random_state)

        # ======================
        # Transform
        # ======================
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        # ======================
        # MLflow
        # ======================
        with mlflow.start_run():

            mlflow.set_tag(
                "run_name",
                f"{vectorizer_name}_ngram_{self.ngram_range}_mf_{max_features}"
            )

            mlflow.log_param("vectorizer", vectorizer_name)
            mlflow.log_param("ngram_range", str(self.ngram_range))
            mlflow.log_param("max_features", max_features)
            mlflow.log_param("model", model_name)

            # ======================
            # Train
            # ======================
            model.fit(X_train_vec, y_train)

            # ======================
            # Predict
            # ======================
            y_pred = model.predict(X_test_vec)

            # ======================
            # Metrics
            # ======================
            acc = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)

            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_macro", report["macro avg"]["f1-score"])
            mlflow.log_metric("precision_macro", report["macro avg"]["precision"])
            mlflow.log_metric("recall_macro", report["macro avg"]["recall"])

            # ======================
            # Confusion Matrix
            # ======================
            cm = confusion_matrix(y_test, y_pred)

            plt.figure(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
            plt.xlabel("Predicted")
            plt.ylabel("Actual")
            plt.title(f"{vectorizer_name} | max_features={max_features}")

            os.makedirs("artifacts/plots", exist_ok=True)
            path = f"artifacts/plots/cm_{vectorizer_name}_{max_features}.png"
            plt.savefig(path)
            plt.close()

            mlflow.log_artifact(path)

            # ======================
            # Model log
            # ======================
            mlflow.sklearn.log_model(model, artifact_path="model")

            print(f"[DONE] max_features={max_features} | accuracy={acc:.4f}")

In [17]:
trainer = ModelTrainer(df_clean)

trainer.config(
    max_features_list=[1000, 2000, 3000, 5000, 8000, 10000],
    model_name="random_forest"
)

2026/05/09 02:24:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:24:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=1000 | accuracy=0.6630
🏃 View run loud-slug-818 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/3d0719e0ef44449598b19bba96c5f1d1
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 02:26:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:26:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=2000 | accuracy=0.6561
🏃 View run stately-squirrel-206 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/48efbfb5219f46c9a30ea50c7fd949b1
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 02:28:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:28:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

[DONE] max_features=5000 | accuracy=0.6521
🏃 View run mercurial-flea-500 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/90afbb00c45d4feab5b453c9ba69af71
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 02:31:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:31:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=8000 | accuracy=0.6493
🏃 View run gregarious-crane-168 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/c5cc1d14628e49e4b498305886eee47a
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 02:33:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:33:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=10000 | accuracy=0.6494
🏃 View run flawless-kit-835 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/cd15dced72ca4a3b845ae361444ec3cb
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


In [20]:
trainer = ModelTrainer(df_clean)

trainer.config(
    max_features_list=[1000, 2000, 3000, 5000, 8000, 10000],
    model_name="xgboost"
)

2026/05/09 02:55:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:55:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=1000 | accuracy=0.7885
🏃 View run resilient-panda-396 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/b007551f1d304af7b013027d8a39dca4
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 02:58:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 02:58:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=2000 | accuracy=0.8049
🏃 View run abrasive-grouse-232 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/48d8d3d7d3ee48fa89e7621a5d61ed59
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 03:03:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 03:03:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

[DONE] max_features=5000 | accuracy=0.8034
🏃 View run gifted-foal-419 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/f20a95372f6b4ab5bf6c8df9d931190c
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 03:12:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 03:12:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=8000 | accuracy=0.8032
🏃 View run rumbling-goose-40 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/fec484150b8c4d119ad39cf96bbee743
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7


2026/05/09 03:17:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/09 03:17:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[DONE] max_features=10000 | accuracy=0.8047
🏃 View run bouncy-rat-4 at: https://mlflow-dashboard.duckdns.org/#/experiments/7/runs/6b559f81f79c47fbb2d86ec977b8b92b
🧪 View experiment at: https://mlflow-dashboard.duckdns.org/#/experiments/7
